# Lesson 1 — How Spikes Become an EEG-Like Signal

**Required · simulation · about 60 minutes**

Follow one simple chain:

**Spike times → responses over time → weighted contributions → their sum.**

Your goal is to explain where each part of the final waveform comes from.
We use the two original functions from `S1/S1_1_EEG_source.ipynb` unchanged.
Every main step is followed immediately by a figure and an observation question.

This is an explanatory model: one synthetic signal at 1000 Hz, in arbitrary
units (a.u.). Its PSP-shaped response is a teaching kernel, not a calibrated
synapse or scalp-voltage model. Real EEG is not simply a smoothed spike trace.

**In VS Code:** select the course kernel and choose **Run All**. Six figures
appear below their code cells. No hardware, downloads, or statistics lab is
needed.


## 1. Start with a small population

We use 20 model neurons so individual events remain visible: 12 produce
periodic spikes, and 8 produce irregular background spikes. First run the
example unchanged. You only need to edit two parameters near the end.


In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "neuradock_eeg101").is_dir())
IN_NOTEBOOK = "ipykernel" in sys.modules
if IN_NOTEBOOK:
    get_ipython().run_line_magic("matplotlib", "inline")
else:
    os.environ.setdefault("MPLBACKEND", "Agg")
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_DIR = ROOT / "outputs" / "notebooks" / "lesson-01" / "walkthrough"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES = []
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

def show_and_save(fig, filename):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches="tight", facecolor="white")
    FIGURES.append(filename)
    if IN_NOTEBOOK:
        plt.show()
    plt.close(fig)

def raster(ax, trains, active_num):
    colors = ["#176B87" if i < active_num else "#C26630" for i in range(len(trains))]
    ax.eventplot(trains, lineoffsets=np.arange(len(trains)),
                 linelengths=0.7, linewidths=1.2, colors=colors)
    ax.axhline(active_num - 0.5, color="gray", lw=0.7, ls=":")
    ax.set(ylim=(-1, len(trains)), ylabel="Neuron index")


In [ ]:
N, ACTIVE_NUM = 20, 12
TARGET_FREQ = 10.0          # Ten periodic spikes per second
PHASE_VARIATION = np.pi / 4 # Across-neuron phase spread
BACKGROUND_FREQ = 5.0      # Background event rate
DURATION, FS = 2.0, 1000
SPIKE_SEED, WEIGHT_SEED = 42, 42
VIEW = (0.4, 1.2)          # Zoom only; the full two-second output is preserved


## 2. When do the neurons spike?

Run the original generator below, then its plotting cell. A vertical tick
means **one event at a particular time**; it is not the shape of an action
potential. Blue neurons repeat a fixed timing pattern. Orange neurons have
irregular intervals. Both groups contribute to the final signal.


In [ ]:
def simulate_spiking_neurons(N, active_num, target_freq, phase_variation, background_freq, duration=1.0, seed=42):
    np.random.seed(seed)
    T = 1.0 / target_freq if target_freq > 0 else float('inf')
    spikes = []
    
    for i in range(N):
        if i < active_num:
            phase = np.random.uniform(-phase_variation, phase_variation)
            n_cycles = int(duration * target_freq)
            base_times = np.arange(n_cycles) * T
            spike_times = base_times + (phase / (2 * np.pi * target_freq))
            spike_times = spike_times[spike_times < duration]
        else:
            isi = np.random.exponential(1/background_freq, size=int(1.5*background_freq*duration))
            spike_times = np.cumsum(isi)
            spike_times = spike_times[spike_times < duration]
        
        spikes.append(spike_times)
    
    return spikes


In [ ]:
spike_trains = simulate_spiking_neurons(
    N, ACTIVE_NUM, TARGET_FREQ, PHASE_VARIATION, BACKGROUND_FREQ,
    duration=DURATION, seed=SPIKE_SEED)

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True,
                          gridspec_kw={"height_ratios": [1, 1, 2.3]})
axes[0].eventplot([spike_trains[0]], lineoffsets=0, colors="#176B87", linelengths=0.8)
axes[0].set(yticks=[], title="One periodic neuron: a spike every 0.1 seconds")
axes[1].eventplot([spike_trains[ACTIVE_NUM]], lineoffsets=0, colors="#C26630", linelengths=0.8)
axes[1].set(yticks=[], title="One background neuron: irregular spike times")
raster(axes[2], spike_trains, ACTIVE_NUM)
axes[2].set(xlim=VIEW, xlabel="Event time (s)", title="All 20 neurons: one row per neuron")
show_and_save(fig, "01-spike-trains.png")


**Look at the figure:** how long is the interval between blue ticks? Why do
the blue population events form bands rather than one perfectly vertical line?
Each periodic neuron keeps its own phase offset throughout the record.

## 3. How do events become a waveform?

Your original `generate_eeg()` performs three operations: place events into
sampled time bins, apply a fixed response to each event, then add the weighted
contributions. Read it once. The next three figures unpack those operations.


In [ ]:
def generate_eeg(spike_trains, duration=1.0, sampling_rate=1000, seed=42):
    # 使用固定的随机种子确保权重一致
    rand_state = np.random.RandomState(seed)
    num_neurons = len(spike_trains)
    t = np.linspace(0, duration, int(sampling_rate*duration), endpoint=False)
    
    weights = rand_state.normal(loc=0.0, scale=1.0, size=num_neurons)
    
    tau = 0.02
    psp_time = np.arange(0, 0.1, 1/sampling_rate)
    psp = psp_time * np.exp(-psp_time/tau)
    psp /= np.max(psp)
    
    eeg = np.zeros_like(t)
    
    for i in range(num_neurons):
        spike_train = np.zeros_like(t)
        indices = np.round(spike_trains[i] * sampling_rate).astype(int)
        indices = indices[indices < len(t)]
        spike_train[indices] = 1.0
        
        contribution = np.convolve(spike_train, psp, mode='full')[:len(t)]
        eeg += weights[i] * contribution
    
    return t, eeg


### 3A. One spike produces one response

Select one event from neuron 0. The fixed PSP-shaped kernel rises and decays
over time. Its weighted version is that neuron's contribution to the output.

For this inspection we temporarily silence other inputs, keeping all 20
list positions and the same weight seed. This calls the original function;
it does not introduce a replacement model or assign a new weight.


In [ ]:
def isolated_neuron_signal(spike_trains, neuron_index, duration, sampling_rate, seed=42):
    """Keep one neuron's events, preserving its index and the full weight draw.

    Empty trains silence other inputs without shortening or reordering the list.
    All rounding, convolution, signed weights, and boundary behavior remain in
    the original generate_eeg function. The supplied trains are never modified.
    """
    isolated = [np.array([], dtype=float) for _ in spike_trains]
    isolated[neuron_index] = np.asarray(spike_trains[neuron_index])
    return generate_eeg(isolated, duration=duration, sampling_rate=sampling_rate, seed=seed)


In [ ]:
weights = np.random.RandomState(WEIGHT_SEED).normal(0.0, 1.0, size=N)
event_time = spike_trains[0][np.argmin(np.abs(spike_trains[0] - 0.6))]
one_event_input = [np.array([], dtype=float) for _ in range(N)]
one_event_input[0] = np.array([event_time])
time_s, one_event_output = isolated_neuron_signal(one_event_input, 0, DURATION, FS, WEIGHT_SEED)
sampled_event_time = np.round(event_time * FS) / FS

# Display the unchanged kernel defined inside generate_eeg().
kernel_time = np.arange(0, 0.1, 1 / FS)
kernel = kernel_time * np.exp(-kernel_time / 0.02)
kernel /= kernel.max()
fig, axes = plt.subplots(3, 1, figsize=(10, 6.4), sharex=True)
axes[0].vlines(sampled_event_time, 0, 1, color="#176B87", lw=2)
axes[0].set(ylim=(-0.1, 1.3), yticks=[0, 1], ylabel="Event", title="One event from neuron 0, rounded to its sample bin")
axes[1].plot(sampled_event_time + kernel_time, kernel, color="#176B87", lw=2)
axes[1].set(ylabel="Kernel (a.u.)", title="The fixed response placed at that event")
axes[2].plot(time_s, one_event_output, color="#C26630", lw=2)
axes[2].set(xlim=(sampled_event_time - 0.03, sampled_event_time + 0.14),
            xlabel="Sample time (s)", ylabel="Contribution (a.u.)",
            title=f"Original function output: the response multiplied by weight {weights[0]:.3f}")
show_and_save(fig, "02-one-spike-response.png")


**Observe:** the event occurs at one time, but its response lasts longer.
The response reaches its peak after the event, not before it.

### 3B. Several responses add up

Now use three early, closely spaced events from background neuron 12 in the
same simulated population. Keep its original weight. Draw each event's
response separately, then their sum. Where responses overlap, they add.
This is the visual meaning of **convolution with the fixed kernel**.


In [ ]:
background_index = ACTIVE_NUM
selected_events = spike_trains[background_index]
selected_events = selected_events[selected_events >= 0.4][:3]
assert len(selected_events) == 3, "This teaching example needs three background events after 0.4 s."
event_responses = []
for event in selected_events:
    event_input = [np.array([], dtype=float) for _ in range(N)]
    event_input[background_index] = np.array([event])
    _, response = isolated_neuron_signal(event_input, background_index, DURATION, FS, WEIGHT_SEED)
    event_responses.append(response)
event_responses = np.array(event_responses)
three_event_input = [np.array([], dtype=float) for _ in range(N)]
three_event_input[background_index] = selected_events
_, three_event_output = isolated_neuron_signal(three_event_input, background_index, DURATION, FS, WEIGHT_SEED)
np.testing.assert_allclose(event_responses.sum(axis=0), three_event_output, atol=1e-12)

fig, axes = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True)
colors = ["#176B87", "#C26630", "#755DA1"]
for number, (event, response, color) in enumerate(zip(selected_events, event_responses, colors), start=1):
    axes[0].vlines(np.round(event * FS) / FS, 0, 1, color=color, lw=2)
    axes[1].plot(time_s, response, color=color, lw=1.8, label=f"Response {number}")
axes[0].set(ylim=(-0.1, 1.3), ylabel="Events", yticks=[0, 1], title="Three selected events from the same background neuron")
axes[1].set(ylabel="Contribution (a.u.)", title="One weighted response per event")
axes[1].legend(fontsize=9)
axes[2].plot(time_s, three_event_output, color="#176B87", lw=2)
axes[2].set(xlim=(selected_events[0] - 0.03, selected_events[-1] + 0.14),
            xlabel="Sample time (s)", ylabel="Sum (a.u.)", title="Their sum equals the original function's three-event output")
show_and_save(fig, "03-responses-add-up.png")


**Question:** point to a time when two responses overlap. How does the bottom
trace relate to the two values directly above it?

### 3C. The population contributions form the final signal

Return to all original events. Isolate each neuron in turn, without changing
its index or weight, and add the resulting curves. The original weights have
both signs, so some contributions reinforce and others cancel.

We show four example contributions and then the sum of **all 20**, not just
those four. The check below confirms that this decomposition reproduces the
original full output. Weight signs here are mathematical coefficients, not
labels for excitatory or inhibitory neurons.


In [ ]:
time_s, eeg = generate_eeg(spike_trains, duration=DURATION, sampling_rate=FS, seed=WEIGHT_SEED)
contributions = np.array([
    isolated_neuron_signal(spike_trains, i, DURATION, FS, WEIGHT_SEED)[1]
    for i in range(N)
])
np.testing.assert_allclose(contributions.sum(axis=0), eeg, atol=1e-12)

displayed_neurons = [0, 1, ACTIVE_NUM, ACTIVE_NUM + 1]
fig, axes = plt.subplots(5, 1, figsize=(11, 9), sharex=True,
                          gridspec_kw={"height_ratios": [1, 1, 1, 1, 1.7]})
for ax, i in zip(axes[:4], displayed_neurons):
    color = "#176B87" if weights[i] >= 0 else "#C26630"
    ax.plot(time_s, contributions[i], color=color, lw=1.6)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_ylabel("a.u.")
    ax.set_title(f"Neuron {i}: {'periodic' if i < ACTIVE_NUM else 'background'}, weight {weights[i]:+.3f}", loc="left", fontsize=10)
axes[-1].plot(time_s, eeg, color="#176B87", lw=1.8, label="Original generate_eeg output")
axes[-1].plot(time_s, contributions.sum(axis=0), "--", color="#C26630", lw=1, label="Sum of all 20 contributions")
axes[-1].set(xlim=VIEW, xlabel="Sample time (s)", ylabel="EEG-like sum (a.u.)",
             title="All 20 contributions add up to the original output")
axes[-1].legend(fontsize=8)
show_and_save(fig, "04-weighted-population.png")
print("The sum of all isolated contributions matches the unchanged original output.")


**Explain the chain:** choose one event, describe its response, explain its
weight, and say how it enters the bottom curve. That is the main learning goal.

## 4. Try two small changes

### A. Change the periodic event frequency

Compare 5 Hz with 10 Hz while keeping the other arguments and seeds fixed.
First predict the interval between events. Then compare the raster and output.
Frequency changes the number of periodic events as well as their spacing.
We do not need a PSD calculation to see this first timing relationship.


In [ ]:
frequency_values = [5.0, 10.0]
frequency_cases = []
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True, sharey="row")
for column, frequency in enumerate(frequency_values):
    trains = simulate_spiking_neurons(N, ACTIVE_NUM, frequency, PHASE_VARIATION,
                                      BACKGROUND_FREQ, duration=DURATION, seed=SPIKE_SEED)
    t_case, signal_case = generate_eeg(trains, duration=DURATION, sampling_rate=FS, seed=WEIGHT_SEED)
    frequency_cases.append(signal_case)
    raster(axes[0, column], trains, ACTIVE_NUM)
    axes[0, column].set_title(f"{frequency:g} Hz: one periodic event every {1 / frequency:g} s")
    axes[1, column].plot(t_case, signal_case, color="#176B87", lw=1.4)
    axes[1, column].set(xlim=VIEW, xlabel="Time (s)", ylabel="EEG-like sum (a.u.)")
fig.suptitle("Only target_freq changes; the kernel and signed weights stay the same")
show_and_save(fig, "05-frequency-comparison.png")


**Observe:** where do the periodic event bands become closer together? Does
the output still contain variation from background events?

### B. Change when neurons spike relative to one another

Compare a phase half-width of 0 with π. Each periodic neuron still repeats
every 0.1 s, but neurons no longer fire at the same moment. Look at the raster
first, then the waveform, using the same axis scale for both conditions.

The weights are signed: **more synchronous events do not guarantee a larger
output amplitude in this model**. Do not change the weights to force a trend.


In [ ]:
phase_values = [0.0, np.pi]
phase_cases = []
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True, sharey="row")
for column, (spread, label) in enumerate(zip(phase_values, ["Same phase", "Dispersed phases"])):
    trains = simulate_spiking_neurons(N, ACTIVE_NUM, TARGET_FREQ, spread,
                                      BACKGROUND_FREQ, duration=DURATION, seed=SPIKE_SEED)
    t_case, signal_case = generate_eeg(trains, duration=DURATION, sampling_rate=FS, seed=WEIGHT_SEED)
    phase_cases.append(signal_case)
    raster(axes[0, column], trains, ACTIVE_NUM)
    axes[0, column].set_title(label)
    axes[1, column].plot(t_case, signal_case, color="#176B87", lw=1.4)
    axes[1, column].set(xlim=VIEW, xlabel="Time (s)", ylabel="EEG-like sum (a.u.)")
fig.suptitle("Only phase_variation changes; the periodic rate stays at 10 Hz")
show_and_save(fig, "06-phase-comparison.png")


## 5. Explain the process in your own words

You have completed the main lesson when you can answer:

1. What does one tick in a spike train represent?
2. Why can its contribution last longer than the event itself?
3. What happens when responses overlap?
4. Why can two neurons' weighted contributions cancel?
5. How is the final curve related to the individual contributions?

**Submit:** the population-contribution figure and five sentences explaining
the chain. No parameter sweep or statistical report is required here.

The next cell only saves your work. Figures and full two-second arrays go to
`outputs/notebooks/lesson-01/walkthrough/`; the plotted zoom does not crop or
repair the original output.


In [ ]:
np.savez_compressed(OUTPUT_DIR / "walkthrough.npz",
    time_s=time_s, eeg_au=eeg, weights=weights, contributions_au=contributions,
    spike_times_s=np.concatenate(spike_trains),
    spike_offsets=np.r_[0, np.cumsum([len(train) for train in spike_trains])],
    selected_events_s=selected_events, event_responses_au=event_responses,
    three_event_output_au=three_event_output,
    one_event_time_s=event_time, one_event_output_au=one_event_output,
    frequency_outputs_au=np.array(frequency_cases), phase_outputs_au=np.array(phase_cases))
summary = {
    "mode": "simulation", "amplitude_unit": "a.u.", "output_channels": 1,
    "parameters": dict(N=N, active_num=ACTIVE_NUM, target_freq=TARGET_FREQ,
                       phase_variation=PHASE_VARIATION, background_freq=BACKGROUND_FREQ,
                       duration=DURATION, sampling_rate=FS),
    "spike_seed": SPIKE_SEED, "weight_seed": WEIGHT_SEED,
    "display_window_s": list(VIEW), "raw_shape": list(eeg.shape),
    "frequency_values_hz": frequency_values, "phase_values_rad": phase_values,
    "contribution_sum_max_error": float(np.max(np.abs(contributions.sum(axis=0) - eeg))),
    "event_sum_max_error": float(np.max(np.abs(event_responses.sum(axis=0) - three_event_output))),
    "negative_event_count": int(sum(np.count_nonzero(train < 0) for train in spike_trains)),
    "selected_background_neuron": background_index,
    "selected_events_s": selected_events.tolist(), "figures": FIGURES,
    "source": json.loads((ROOT / "docs" / "spike2eeg-source.json").read_text(encoding="utf-8")),
    "limitations": ["Teaching model, not a head model or calibrated scalp EEG.",
                    "Original negative-time indexing and finite background interval batches are preserved.",
                    "The displayed interior interval is a plot zoom, not a modification of raw outputs."]
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")
print(f"Saved {len(FIGURES)} figures and the original full-length arrays to {OUTPUT_DIR}")


**Next:** [Lesson 2](02-reading-neuradock-data.ipynb) introduces the actual
seven-channel, 250 Hz NeuraDock recording format. That device profile is
distinct from this one-signal, arbitrary-unit simulation.
